# Adversarial KD MNIST — Repo-Ready Multi-Seed Experiment

This notebook runs the MNIST adversarial knowledge distillation experiment across multiple random seeds.

**Important:** This version is intended to live inside the final project repository. It does **not** clone the GitHub repository automatically. Before running, make sure the repo contains:

```text
src/adversarial_kd_mnist.py
requirements.txt
```

Recommended Colab setting: `Runtime → Change runtime type → GPU`. A100 is recommended for the full 3-seed run.


In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ── Cell 2: Locate current repo & install dependencies ──────────────────────
# This repo-ready version does NOT clone from GitHub.
# It expects this notebook to be placed inside the final project repo,
# or run from a working directory that contains src/adversarial_kd_mnist.py.

import os
import sys
import subprocess
from pathlib import Path

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()

    # 1) Search current directory and its parents.
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'adversarial_kd_mnist.py').exists():
            return candidate

    # 2) Common locations when the repo is manually uploaded or mounted in Colab.
    common_candidates = [
        Path('/content/ML_Project'),
        Path('/content/drive/MyDrive/ML_Project'),
    ]
    for candidate in common_candidates:
        if (candidate / 'src' / 'adversarial_kd_mnist.py').exists():
            return candidate.resolve()

    raise FileNotFoundError(
        'Could not find src/adversarial_kd_mnist.py. '
        'Run this notebook from the project repo root, place it under the repo, '
        'or upload/mount the full repo before running. This version intentionally does not clone the repo.'
    )

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

requirements_path = REPO_ROOT / 'requirements.txt'
if requirements_path.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)])
else:
    print('requirements.txt not found; assuming dependencies are already installed.')

print(f'Repo root: {REPO_ROOT}')
print('Setup done.')


In [ ]:
# ── Cell 3: Imports ──────────────────────────────────────────────────────────
import os
import sys
import json
from pathlib import Path
from datetime import datetime
from dataclasses import asdict, replace

import numpy as np
import pandas as pd

# If this cell is run independently, try to recover REPO_ROOT from the current path.
if 'REPO_ROOT' not in globals():
    REPO_ROOT = Path.cwd().resolve()

if not (Path(REPO_ROOT) / 'src' / 'adversarial_kd_mnist.py').exists():
    raise FileNotFoundError(
        f'Cannot find src/adversarial_kd_mnist.py under {REPO_ROOT}. '
        'Please run Cell 2 first, or start this notebook from the project repo root.'
    )

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.adversarial_kd_mnist import (
    Config,
    EPSILONS,
    SmallCNN,
    set_seed,
    get_device,
    get_loaders,
    accuracy,
    fgsm_attack,
    pgd_attack,
    evaluate_under_attack,
    train_supervised,
    train_standard_kd,
    train_adversarial,
    train_adversarial_kd,
    save_checkpoint,
    save_training_losses_csv,
    plot_training_losses,
    save_results_csv,
    plot_curves,
    visualize_adversarial_examples,
)

print('Imports OK.')


In [ ]:
# ── Cell 4: Experiment settings ──────────────────────────────────────────────
# Adjust NUM_RUNS / epochs as needed.
# With A100: 3 runs × 15 epochs ≈ 15–25 min

NUM_RUNS     = 3
EVAL_REPEATS = 3

config = Config(
    data_dir="data",
    output_dir="outputs",
    run_name="multi_seed_repeated",
    epochs=15,
    batch_size=128,
    lr=1e-3,
    seed=42,
    temperature=5.0,
    kd_alpha=0.5,
    adv_weight=0.5,
    pgd_steps=40,
    pgd_step_size=0.01,
    adv_train_epsilon=0.1,
    quick_test=False,
)

print(f"NUM_RUNS={NUM_RUNS}, EVAL_REPEATS={EVAL_REPEATS}")
print(f"Seeds : {[config.seed + i for i in range(NUM_RUNS)]}")
print(f"Epochs: {config.epochs}")

In [ ]:
# ── Cell 5: Helper functions ─────────────────────────────────────────────────

def evaluate_under_attack_repeated(model, loader, device, attack_fn,
                                   epsilons, repeats=3, base_seed=42):
    repeated_results = []
    for r in range(repeats):
        set_seed(base_seed + r)
        result = evaluate_under_attack(
            model=model, loader=loader, device=device,
            attack_fn=attack_fn, epsilons=epsilons,
        )
        repeated_results.append(result)
    averaged_results = {}
    for eps in epsilons:
        values = [result[eps] for result in repeated_results]
        averaged_results[eps] = float(np.mean(values))
    return averaged_results


def run_one_seed(config, device, output_root, run_idx, seed):
    print(f"\n========== Run {run_idx + 1}/{NUM_RUNS}, seed={seed} ==========")

    run_config = replace(config, seed=seed)
    set_seed(seed)
    train_loader, test_loader = get_loaders(run_config)

    run_dir = output_root / f"run_{run_idx + 1}_seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    with (run_dir / "config.json").open("w", encoding="utf-8") as f:
        json.dump(asdict(run_config), f, indent=2)

    teacher      = SmallCNN(dropout=0.25).to(device)
    baseline     = SmallCNN(dropout=0.25).to(device)
    standard_kd  = SmallCNN(dropout=0.25).to(device)
    adv_training = SmallCNN(dropout=0.25).to(device)
    adv_kd       = SmallCNN(dropout=0.25).to(device)

    losses_by_model = {}

    print("\nTraining teacher")
    losses_by_model["teacher"] = train_supervised(
        teacher, train_loader, device, run_config, name="teacher")
    save_checkpoint(teacher, run_dir, "teacher")

    print("\nTraining baseline")
    losses_by_model["baseline"] = train_supervised(
        baseline, train_loader, device, run_config, name="baseline")
    save_checkpoint(baseline, run_dir, "baseline")

    print("\nTraining standard KD student")
    losses_by_model["standard_kd"] = train_standard_kd(
        standard_kd, teacher, train_loader, device, run_config)
    save_checkpoint(standard_kd, run_dir, "standard_kd")

    print("\nTraining adversarial training model")
    losses_by_model["adv_training"] = train_adversarial(
        adv_training, train_loader, device, run_config)
    save_checkpoint(adv_training, run_dir, "adv_training")

    print("\nTraining adversarial KD student")
    losses_by_model["adv_kd"] = train_adversarial_kd(
        adv_kd, teacher, train_loader, device, run_config)
    save_checkpoint(adv_kd, run_dir, "adv_kd")

    save_training_losses_csv(losses_by_model, run_dir)
    plot_training_losses(losses_by_model, run_dir)

    models = {
        "baseline":     baseline,
        "standard_kd":  standard_kd,
        "adv_training": adv_training,
        "adv_kd":       adv_kd,
    }

    pgd_fn = lambda m, x, y, eps: pgd_attack(
        m, x, y, epsilon=eps,
        steps=run_config.pgd_steps, step_size=run_config.pgd_step_size,
    )

    results = {}
    for model_name, model in models.items():
        clean_acc = accuracy(model, test_loader, device)
        print(f"\n{model_name}: clean accuracy = {clean_acc:.4f}")

        results[model_name] = {
            "clean": {0.0: clean_acc},
            "fgsm": evaluate_under_attack_repeated(
                model=model, loader=test_loader, device=device,
                attack_fn=fgsm_attack, epsilons=EPSILONS,
                repeats=EVAL_REPEATS, base_seed=seed + 100,
            ),
            "pgd": evaluate_under_attack_repeated(
                model=model, loader=test_loader, device=device,
                attack_fn=pgd_fn, epsilons=EPSILONS,
                repeats=EVAL_REPEATS, base_seed=seed + 1000,
            ),
        }

    save_results_csv(results, run_dir)
    plot_curves(results, run_dir)
    visualize_adversarial_examples(baseline, test_loader, device, run_config, run_dir)

    return results


def summarize_runs(all_results, output_root):
    rows = []
    model_names = all_results[0].keys()
    for model_name in model_names:
        for attack_name in all_results[0][model_name].keys():
            for eps in all_results[0][model_name][attack_name].keys():
                values = [r[model_name][attack_name][eps] for r in all_results]
                mean_acc = float(np.mean(values))
                std_acc  = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
                rows.append({
                    "model": model_name, "attack": attack_name,
                    "epsilon": eps, "mean_accuracy": mean_acc, "std_accuracy": std_acc,
                })

    summary_df = pd.DataFrame(rows)
    summary_path = output_root / "summary_mean_std.csv"
    summary_df.to_csv(summary_path, index=False)

    print("\n========== Summary: mean ± std over training seeds ==========")
    display(summary_df)
    print(f"\nSaved summary to: {summary_path}")
    return summary_df

print("Helper functions defined.")

In [ ]:
# ── Cell 6: Run experiment ───────────────────────────────────────────────────

device = get_device()
print(f"Using device: {device}")

timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
output_root = Path(config.output_dir) / f"{config.run_name}_{timestamp}"
output_root.mkdir(parents=True, exist_ok=True)

with (output_root / "multi_seed_config.json").open("w", encoding="utf-8") as f:
    json.dump({
        **asdict(config),
        "num_runs":    NUM_RUNS,
        "eval_repeats": EVAL_REPEATS,
        "seeds":       [config.seed + i for i in range(NUM_RUNS)],
    }, f, indent=2)

all_results = []
for run_idx in range(NUM_RUNS):
    seed = config.seed + run_idx
    run_result = run_one_seed(
        config=config, device=device,
        output_root=output_root, run_idx=run_idx, seed=seed,
    )
    all_results.append(run_result)

summary_df = summarize_runs(all_results, output_root)
print(f"\nDone. All results saved to: {output_root.resolve()}")

In [ ]:
# ── Cell 7: Optional: package results as zip ────────────────────────────────
# Run this cell after the experiment finishes if you want a zip file of the outputs.

import shutil
from pathlib import Path

zip_path = Path.cwd() / f'{output_root.name}.zip'
shutil.make_archive(str(zip_path).replace('.zip', ''), 'zip', str(output_root))
print(f'Created zip file: {zip_path}')

# In Google Colab, this will trigger a browser download.
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print('Not running in Colab, or automatic download is unavailable. Use the printed zip path above.')
